In [0]:
# Databricks notebook source
# =============================================================================
# LMS LEARNING TRANSACTION — BRONZE → SILVER + QUARANTINE
# Catalog : hackathon_ltm
# Bronze  : hackathon_ltm.bronze.lms_learning_transaction          (source)
# Silver  : hackathon_ltm.silver.silver_lms_learning_transaction   (clean)
# Quarantine: hackathon_ltm.quarantine.quarantine_lms_learning_transaction
# =============================================================================

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 0 ▶ IMPORTS & CONFIGURATION
# ─────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DateType, StringType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

CATALOG         = "hackathon_ltm"
BRONZE_SCHEMA   = "bronze"
SILVER_SCHEMA   = "silver"
QUARANTINE_SCHEMA = "quarantine"

BRONZE_TABLE      = f"{CATALOG}.{BRONZE_SCHEMA}.lms_learning_transaction"
SILVER_TABLE      = f"{CATALOG}.{SILVER_SCHEMA}.silver_lms_learning_transaction"
QUARANTINE_TABLE  = f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_lms_learning_transaction"

print(f"Source      : {BRONZE_TABLE}")
print(f"Silver      : {SILVER_TABLE}")
print(f"Quarantine  : {QUARANTINE_TABLE}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 1 ▶ READ FROM BRONZE DELTA TABLE
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 1: Reading Bronze Delta Table")
print("="*60)

df_raw = spark.table(BRONZE_TABLE)
total_raw = df_raw.count()

print(f"  Total rows in bronze table : {total_raw}")
df_raw.printSchema()
df_raw.show(5, truncate=False)

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 2 ▶ STRUCTURAL FIX — EXTRA COLUMN / MALFORMED ROWS
#
#  ROOT CAUSE: 'Requested' status rows have a spurious trailing
#  comma in the CSV source, producing 13 fields instead of 12.
#  In Delta this may manifest as a null/empty extra column.
#  We drop any unnamed/extra columns here to normalise the schema.
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 2: Structural Fix — Drop Spurious Extra Columns")
print("="*60)

EXPECTED_COLS = [
    "learning_id", "employee_id", "course_id", "course_request_id",
    "request_date", "approval_status", "approval_date", "enrollment_date",
    "start_date", "completion_date", "score", "passing_score"
]

# Keep only expected columns (handles any extra blank column from CSV ingestion)
df = df_raw.select(*[c for c in EXPECTED_COLS if c in df_raw.columns])
print(f"  Columns retained: {df.columns}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 3 ▶ DATA TYPE CASTING
#
#  All columns arrive as STRING from CSV ingestion.
#  Cast to proper types before any validation logic.
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 3: Data Type Casting")
print("="*60)

date_cols = ["request_date", "approval_date", "enrollment_date",
             "start_date", "completion_date"]

# Replace empty string with NULL for all columns (blanks ingested as "")
for col in df.columns:
    df = df.withColumn(col, F.when(F.trim(F.col(col)) == "", None).otherwise(F.col(col)))

# Cast date columns
for col in date_cols:
    df = df.withColumn(col, F.to_date(F.col(col), "yyyy-MM-dd"))

# Cast numeric columns
df = (df
      .withColumn("score",         F.col("score").cast(IntegerType()))
      .withColumn("passing_score", F.col("passing_score").cast(IntegerType()))
)

# Trim whitespace on all string columns
string_cols = ["learning_id", "employee_id", "course_id",
               "course_request_id", "approval_status"]
for col in string_cols:
    df = df.withColumn(col, F.trim(F.col(col)))

print("  Schema after casting:")
df.printSchema()

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 4 ▶ STANDARDISE approval_status (Title-Case)
#
#  Ensure consistent casing: Approved / Rejected / Requested
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 4: Standardise approval_status")
print("="*60)

df = df.withColumn("approval_status",
                   F.initcap(F.lower(F.col("approval_status"))))

print("  approval_status distribution:")
df.groupBy("approval_status").count().orderBy("count", ascending=False).show()

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 5 ▶ QUARANTINE FLAG LOGIC
#
#  Tag each row with a quarantine_reason. A row is quarantined
#  (sent to quarantine table) if ANY rule fires.
#  Clean rows (no rule fired) go to silver.
#
#  RULE 1  — NULL primary key     : learning_id IS NULL
#  RULE 2  — NULL business key    : employee_id OR course_id IS NULL
#  RULE 3  — Invalid status       : approval_status NOT IN allowed set
#  RULE 4  — Approved missing dates: Approved rows with NULL
#                                    enrollment_date OR start_date
#  RULE 5  — Date sequence violation:
#             approval_date  < request_date   (impossible)
#             enrollment_date < approval_date  (impossible)
#             start_date     < enrollment_date (impossible)
#             completion_date < start_date     (impossible)
#  RULE 6  — Score without completion_date (data inconsistency)
#  RULE 7  — Completion without score (data inconsistency)
#  RULE 8  — Score out of range (score < 0 OR score > 100)
#  RULE 9  — Duplicate learning_id (keep first, quarantine rest)
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 5: Applying Quarantine Flag Rules")
print("="*60)

VALID_STATUSES = ["Approved", "Rejected", "Requested"]

# ── RULE 1: NULL primary key
r1 = F.when(F.col("learning_id").isNull(), "RULE1_NULL_LEARNING_ID")

# ── RULE 2: NULL business keys
r2 = F.when(F.col("employee_id").isNull() | F.col("course_id").isNull(),
            "RULE2_NULL_EMPLOYEE_OR_COURSE")

# ── RULE 3: Invalid approval_status
r3 = F.when(~F.col("approval_status").isin(VALID_STATUSES),
            "RULE3_INVALID_APPROVAL_STATUS")

# ── RULE 4: Approved rows missing enrollment or start date
r4 = F.when(
    (F.col("approval_status") == "Approved") &
    (F.col("enrollment_date").isNull() | F.col("start_date").isNull()),
    "RULE4_APPROVED_MISSING_ENROLLMENT_OR_START"
)

# ── RULE 5: Date sequence violations
r5 = F.when(
    (F.col("approval_date").isNotNull()   & F.col("request_date").isNotNull()    & (F.col("approval_date")   < F.col("request_date")))   |
    (F.col("enrollment_date").isNotNull() & F.col("approval_date").isNotNull()   & (F.col("enrollment_date") < F.col("approval_date")))   |
    (F.col("start_date").isNotNull()      & F.col("enrollment_date").isNotNull() & (F.col("start_date")      < F.col("enrollment_date"))) |
    (F.col("completion_date").isNotNull() & F.col("start_date").isNotNull()      & (F.col("completion_date") < F.col("start_date"))),
    "RULE5_DATE_SEQUENCE_VIOLATION"
)

# ── RULE 6: Score present but completion_date missing (inconsistent)
r6 = F.when(
    F.col("score").isNotNull() & F.col("completion_date").isNull(),
    "RULE6_SCORE_WITHOUT_COMPLETION_DATE"
)

# ── RULE 7: Completion date present but score missing
r7 = F.when(
    F.col("completion_date").isNotNull() & F.col("score").isNull(),
    "RULE7_COMPLETION_DATE_WITHOUT_SCORE"
)

# ── RULE 8: Score out of valid range
r8 = F.when(
    F.col("score").isNotNull() & ((F.col("score") < 0) | (F.col("score") > 100)),
    "RULE8_SCORE_OUT_OF_RANGE"
)

# ── RULE 9: Duplicate learning_id — mark all but the first occurrence
w_dedup = Window.partitionBy("learning_id").orderBy("request_date")
df = df.withColumn("_row_num", F.row_number().over(w_dedup))
r9 = F.when(F.col("_row_num") > 1, "RULE9_DUPLICATE_LEARNING_ID")

# ── Combine all rules into a single quarantine_reason column
df = df.withColumn("quarantine_reason",
    F.coalesce(r1, r2, r3, r4, r5, r6, r7, r8, r9)
)

# Summary of flagged rows
print("  Quarantine rule breakdown:")
(df.filter(F.col("quarantine_reason").isNotNull())
   .groupBy("quarantine_reason").count()
   .orderBy("count", ascending=False)
   .show(truncate=False))

total_clean = df.filter(F.col("quarantine_reason").isNull()).count()
total_quarantine = df.filter(F.col("quarantine_reason").isNotNull()).count()
print(f"  Clean rows     : {total_clean}")
print(f"  Quarantine rows: {total_quarantine}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 6 ▶ SPLIT INTO CLEAN + QUARANTINE DATAFRAMES
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 6: Splitting into Clean and Quarantine DataFrames")
print("="*60)

df_quarantine = (df
    .filter(F.col("quarantine_reason").isNotNull())
    .drop("_row_num")
    .withColumn("quarantine_timestamp", F.current_timestamp())
    .withColumn("source_table", F.lit(BRONZE_TABLE))
)

df_clean = (df
    .filter(F.col("quarantine_reason").isNull())
    .drop("quarantine_reason", "_row_num")
)

print(f"  Clean DataFrame     rows: {df_clean.count()}")
print(f"  Quarantine DataFrame rows: {df_quarantine.count()}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 7 ▶ SILVER TRANSFORMATIONS ON CLEAN DATA
#
#  7a. Derive: completion_status
#       - 'Completed' if completion_date is not null
#       - 'In Progress' if approved, started, but no completion
#       - 'Enrolled' if approved, no start yet
#       - 'Pending Approval' if Requested
#       - 'Rejected' if Rejected
#
#  7b. Derive: is_passed (boolean)
#       - TRUE if score >= passing_score
#       - FALSE if score < passing_score
#       - NULL if no score yet
#
#  7c. Derive: score_gap
#       - score - passing_score  (positive = passed margin)
#
#  7d. Derive: course_domain  (from course_id suffix)
#       - _DB → 'Data & BI'
#       - _MS → 'Microsoft / Office'
#       - _CR → 'Core / Compliance'
#
#  7e. Derive: days_to_approve  (approval_date - request_date)
#  7f. Derive: days_to_enroll   (enrollment_date - approval_date)
#  7g. Derive: days_to_start    (start_date - enrollment_date)
#  7h. Derive: days_to_complete (completion_date - start_date)
#  7i. Derive: total_learning_days (completion_date - start_date)
#
#  7j. Metadata columns:
#       - ingestion_timestamp (current time)
#       - source_table
#       - processing_date (today's date)
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 7: Silver Transformations & Derived Columns")
print("="*60)

# 7a — completion_status
df_silver = df_clean.withColumn(
    "completion_status",
    F.when(F.col("completion_date").isNotNull(), "Completed")
     .when(
         (F.col("approval_status") == "Approved") &
         F.col("start_date").isNotNull() &
         F.col("completion_date").isNull(),
         "In Progress"
     )
     .when(
         (F.col("approval_status") == "Approved") &
         F.col("enrollment_date").isNotNull() &
         F.col("start_date").isNull(),
         "Enrolled"
     )
     .when(F.col("approval_status") == "Requested",  "Pending Approval")
     .when(F.col("approval_status") == "Rejected",   "Rejected")
     .otherwise("Unknown")
)
print("  ✅ 7a: completion_status derived")

# 7b — is_passed
df_silver = df_silver.withColumn(
    "is_passed",
    F.when(
        F.col("score").isNotNull() & F.col("passing_score").isNotNull(),
        F.col("score") >= F.col("passing_score")
    ).otherwise(None)
)
print("  ✅ 7b: is_passed derived")

# 7c — score_gap
df_silver = df_silver.withColumn(
    "score_gap",
    F.when(
        F.col("score").isNotNull() & F.col("passing_score").isNotNull(),
        F.col("score") - F.col("passing_score")
    ).otherwise(None)
)
print("  ✅ 7c: score_gap derived")

# 7d — course_domain (extracted from the suffix in course_id)
df_silver = df_silver.withColumn(
    "course_domain",
    F.when(F.col("course_id").endswith("_DB"), "Data & BI")
     .when(F.col("course_id").endswith("_MS"), "Microsoft / Office")
     .when(F.col("course_id").endswith("_CR"), "Core / Compliance")
     .otherwise("Unknown")
)
print("  ✅ 7d: course_domain derived")

# 7e — days_to_approve
df_silver = df_silver.withColumn(
    "days_to_approve",
    F.when(
        F.col("approval_date").isNotNull() & F.col("request_date").isNotNull(),
        F.datediff(F.col("approval_date"), F.col("request_date"))
    ).otherwise(None)
)
print("  ✅ 7e: days_to_approve derived")

# 7f — days_to_enroll
df_silver = df_silver.withColumn(
    "days_to_enroll",
    F.when(
        F.col("enrollment_date").isNotNull() & F.col("approval_date").isNotNull(),
        F.datediff(F.col("enrollment_date"), F.col("approval_date"))
    ).otherwise(None)
)
print("  ✅ 7f: days_to_enroll derived")

# 7g — days_to_start
df_silver = df_silver.withColumn(
    "days_to_start",
    F.when(
        F.col("start_date").isNotNull() & F.col("enrollment_date").isNotNull(),
        F.datediff(F.col("start_date"), F.col("enrollment_date"))
    ).otherwise(None)
)
print("  ✅ 7g: days_to_start derived")

# 7h — days_to_complete
df_silver = df_silver.withColumn(
    "days_to_complete",
    F.when(
        F.col("completion_date").isNotNull() & F.col("start_date").isNotNull(),
        F.datediff(F.col("completion_date"), F.col("start_date"))
    ).otherwise(None)
)
print("  ✅ 7h: days_to_complete derived")

# 7i — total_learning_days (request to completion)
df_silver = df_silver.withColumn(
    "total_learning_days",
    F.when(
        F.col("completion_date").isNotNull() & F.col("request_date").isNotNull(),
        F.datediff(F.col("completion_date"), F.col("request_date"))
    ).otherwise(None)
)
print("  ✅ 7i: total_learning_days derived")

# 7j — metadata columns
df_silver = (df_silver
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_table",        F.lit(BRONZE_TABLE))
    .withColumn("processing_date",     F.current_date())
)
print("  ✅ 7j: Metadata columns added")

print("\n  Final Silver Schema:")
df_silver.printSchema()

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 8 ▶ FINAL COLUMN ORDERING
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 8: Final Column Ordering")
print("="*60)

SILVER_COLUMNS = [
    # ── Keys
    "learning_id", "employee_id", "course_id", "course_request_id",
    # ── Status / classification
    "approval_status", "completion_status", "is_passed",
    # ── Course dimension shortcut
    "course_domain",
    # ── Raw dates
    "request_date", "approval_date", "enrollment_date",
    "start_date", "completion_date",
    # ── Scores
    "score", "passing_score", "score_gap",
    # ── Duration KPIs
    "days_to_approve", "days_to_enroll", "days_to_start",
    "days_to_complete", "total_learning_days",
    # ── Metadata
    "ingestion_timestamp", "source_table", "processing_date"
]

df_silver = df_silver.select(*SILVER_COLUMNS)
print(f"  Silver columns ({len(SILVER_COLUMNS)}): {SILVER_COLUMNS}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 9 ▶ DATA QUALITY SUMMARY (pre-write)
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 9: Pre-Write Data Quality Summary")
print("="*60)

print("\n  [ Silver ] completion_status distribution:")
df_silver.groupBy("completion_status").count().orderBy("count", ascending=False).show()

print("\n  [ Silver ] course_domain distribution:")
df_silver.groupBy("course_domain").count().orderBy("count", ascending=False).show()

print("\n  [ Silver ] is_passed distribution:")
df_silver.groupBy("is_passed").count().show()

print("\n  [ Silver ] score statistics:")
df_silver.select("score", "passing_score", "score_gap").summary().show()

print("\n  [ Silver ] Sample rows:")
df_silver.show(5, truncate=False)

print("\n  [ Quarantine ] quarantine_reason distribution:")
df_quarantine.groupBy("quarantine_reason").count().orderBy("count", ascending=False).show(truncate=False)

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 10 ▶ CREATE TARGET SCHEMAS & WRITE SILVER TABLE
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 10: Creating Schemas & Writing Silver Table")
print("="*60)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{QUARANTINE_SCHEMA}")
print("  ✅ Schemas verified / created")

# Write Silver — MERGE (upsert) on learning_id to support incremental runs
if spark.catalog.tableExists(SILVER_TABLE):
    print(f"  Silver table exists — performing MERGE upsert on learning_id")
    silver_delta = DeltaTable.forName(spark, SILVER_TABLE)
    silver_delta.alias("target").merge(
        df_silver.alias("source"),
        "target.learning_id = source.learning_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print(f"  ✅ MERGE complete into {SILVER_TABLE}")
else:
    print(f"  Silver table does not exist — creating with WRITE")
    (df_silver.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"  ✅ Silver table created: {SILVER_TABLE}")

# Optimise and add Z-ORDER for common query patterns
spark.sql(f"OPTIMIZE {SILVER_TABLE} ZORDER BY (employee_id, course_id)")
print(f"  ✅ OPTIMIZE + ZORDER applied on {SILVER_TABLE}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 11 ▶ WRITE QUARANTINE TABLE
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 11: Writing Quarantine Table")
print("="*60)

(df_quarantine.write
    .format("delta")
    .mode("append")          # Append so historical quarantine records are preserved
    .option("mergeSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)
print(f"  ✅ Quarantine table written: {QUARANTINE_TABLE}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 12 ▶ FINAL RECONCILIATION COUNTS
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 12: Final Reconciliation")
print("="*60)

silver_count     = spark.table(SILVER_TABLE).count()
quarantine_count = spark.table(QUARANTINE_TABLE).count()

print(f"  Bronze (source)   : {total_raw:>6} rows")
print(f"  Silver (clean)    : {silver_count:>6} rows")
print(f"  Quarantine        : {quarantine_count:>6} rows")
print(f"  Silver + Quarantine: {silver_count + quarantine_count:>6} rows  "
      f"({'✅ BALANCED' if silver_count + quarantine_count == total_raw else '❌ MISMATCH'})")

print("\n" + "="*60)
print("✅  PIPELINE COMPLETE")
print("="*60)

# COMMAND ----------
# =============================================================================
# QUICK KPI VALIDATION QUERIES (run after pipeline)
# =============================================================================

# -- Completion Rate by Domain
spark.sql(f"""
    SELECT course_domain,
           COUNT(*) AS total,
           SUM(CASE WHEN completion_status = 'Completed' THEN 1 ELSE 0 END) AS completed,
           ROUND(SUM(CASE WHEN completion_status = 'Completed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS completion_rate_pct
    FROM {SILVER_TABLE}
    WHERE approval_status = 'Approved'
    GROUP BY course_domain
    ORDER BY completion_rate_pct DESC
""").show()

# -- Pass Rate by Domain
spark.sql(f"""
    SELECT course_domain,
           COUNT(*) AS assessed,
           SUM(CAST(is_passed AS INT)) AS passed,
           ROUND(SUM(CAST(is_passed AS INT)) * 100.0 / COUNT(*), 2) AS pass_rate_pct,
           ROUND(AVG(score), 2) AS avg_score,
           ROUND(AVG(score_gap), 2) AS avg_score_gap
    FROM {SILVER_TABLE}
    WHERE is_passed IS NOT NULL
    GROUP BY course_domain
    ORDER BY pass_rate_pct DESC
""").show()

# -- Average Turnaround Times
spark.sql(f"""
    SELECT
        ROUND(AVG(days_to_approve),  2) AS avg_days_to_approve,
        ROUND(AVG(days_to_enroll),   2) AS avg_days_to_enroll,
        ROUND(AVG(days_to_start),    2) AS avg_days_to_start,
        ROUND(AVG(days_to_complete), 2) AS avg_days_to_complete,
        ROUND(AVG(total_learning_days), 2) AS avg_total_learning_days
    FROM {SILVER_TABLE}
""").show()

Source      : hackathon_ltm.bronze.lms_learning_transaction
Silver      : hackathon_ltm.silver.silver_lms_learning_transaction
Quarantine  : hackathon_ltm.quarantine.quarantine_lms_learning_transaction

STEP 1: Reading Bronze Delta Table
  Total rows in bronze table : 500
root
 |-- learning_id: string (nullable = true)
 |-- employee_id: string (nullable = true)
 |-- course_id: string (nullable = true)
 |-- course_request_id: string (nullable = true)
 |-- request_date: date (nullable = true)
 |-- approval_status: string (nullable = true)
 |-- approval_date: date (nullable = true)
 |-- enrollment_date: date (nullable = true)
 |-- start_date: date (nullable = true)
 |-- completion_date: date (nullable = true)
 |-- score: integer (nullable = true)
 |-- passing_score: integer (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _source_file: string (nullable = true)

+-----------+-----------+---------+-----------------+-